In [22]:
from agent_framework import Agent, WorkflowBuilder, executor, WorkflowContext, AgentExecutorResponse
from agent_framework.openai import OpenAIChatClient
import os
from dotenv import load_dotenv

load_dotenv()

chat_client = OpenAIChatClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

poem_agent = Agent(
    client=chat_client,
    name="Poem Agent",
    description="An agent that generates poems.",
    instructions="Generate poems based on the given prompts.",
    tools=[],
    context_providers=[],
)

lyric_agent = Agent(
    client=chat_client,
    name="Lyric Agent",
    description="An agent that generates song lyrics.",
    instructions="Generate song lyrics based on the given prompts.",
    tools=[],
    context_providers=[],
)


In [23]:
import logging


@executor(id="upper_case_executor")
async def upper_case(value: AgentExecutorResponse, ctx: WorkflowContext[str, str]) -> None:
# async def upper_case(value: AgentExecutorResponse, ctx: WorkflowContext[str, str]) -> None:
    """Convert the input to uppercase and forward it to the next node."""
    # await ctx.send_message(value.agent_response.text.upper())
    await ctx.send_message(value.agent_response.text.upper())
    await ctx.yield_output("Converting text to uppercase") # WorkflowEvent(type='output', executor_id='upper_case_executor', data='Converting text to uppercase')


@executor(id="reverse_executor")
async def reverse(value: str, ctx: WorkflowContext[str, str]) -> None:
# async def reverse(value: AgentExecutorResponse, ctx: WorkflowContext[str, str]) -> None:
    """Reverse the input text and forward it to the next node."""
    # await ctx.send_message(value.agent_response.text[::-1])
    await ctx.send_message(value[::-1])
    logging.info("Reversing text: %s", value[::-1])
    await ctx.yield_output("Reversing text") # WorkflowEvent(type='output', executor_id='reverse_executor', data='Reversing text')
    await ctx.yield_output("Reversing text 2") # WorkflowEvent(type='output', executor_id='reverse_executor', data='Reversing text')
    await ctx.yield_output("Reversing text 3") # WorkflowEvent(type='output', executor_id='reverse_executor', data='Reversing text')


@executor(id="start_node")
async def start_node(prompt: str, ctx: WorkflowContext[str, str]) -> None:
    """Convert the input to lowercase and forward it to the next node."""
    await ctx.send_message(prompt)
    await ctx.yield_output("Starting execution") # WorkflowEvent(type='output', executor_id='start_node', data='Starting execution')

In [ ]:


from pydantic import JsonValue
from typing import Any
from enum import Enum
from datetime import datetime, date
from uuid import UUID
from pathlib import Path
from dataclasses import is_dataclass, asdict
import json

def make_serializable(value: Any, _seen: set[int] | None = None) -> JsonValue:
    """Best-effort conversion of any Python value to a JSON-serializable form."""
    if value is None or isinstance(value, (bool, int, float, str)):
        return value

    _seen = _seen if _seen is not None else set()
    if id(value) in _seen:  # guard against circular references
        return f"<circular {type(value).__name__}>"

    if isinstance(value, Enum):
        return make_serializable(value.value, _seen)
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, (UUID, Path)):
        return str(value)
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    _seen = _seen | {id(value)}  # per-branch, so shared (non-circular) refs are fine

    if isinstance(value, dict):
        return {str(k): make_serializable(v, _seen) for k, v in value.items()}
    if isinstance(value, (list, tuple, set, frozenset)):
        return [make_serializable(item, _seen) for item in value]
    if is_dataclass(value) and not isinstance(value, type):
        return make_serializable(asdict(value), _seen)
    if hasattr(value, "to_json"):
        method = getattr(value, "to_json", None)
        if callable(method):
            try:
                json_str = method()
                if type(json_str) is str:
                    return json.loads(json_str)
            except Exception:  # skip if fails
                pass
    # Pydantic v2, Pydantic v1, then generic to_dict()
    for method_name in ("model_dump", "dict", "to_dict"):
        method = getattr(value, method_name, None)
        if callable(method):
            try:
                converted = method()
            except Exception:  # try next method if this one fails
                continue
            return make_serializable(converted, _seen)
    # a catch-call for objects with __dict__ but no serialization methods
    if hasattr(value, "__dict__"):
        return make_serializable(vars(value), _seen)
    return str(value)

In [25]:
import json

from agent_framework import WorkflowErrorDetails
from evaluation_utils import capture_logs, capture_spans
import logging

# logging.disable(logging.CRITICAL)
events = []
try:
    with (
        capture_spans() as span_capture,
        capture_logs() as log_capture
    ):
        workflow = (
            WorkflowBuilder(start_executor=start_node)
            .add_fan_out_edges(start_node, [poem_agent, lyric_agent])
            .add_edge(poem_agent, upper_case)
            .add_edge(lyric_agent, upper_case)
            .add_edge(upper_case, reverse)
        ).build()
        async for event in workflow.run("Something short about a hot dog", stream=True):
            # print(type(event))
            # if event.type == "executor_failed" and isinstance(event.data, WorkflowErrorDetails):
            #     print("Executor failed with details:", event.data)
            #     continue
            print(event)
            events.append(event)
except Exception as e:
    print("Workflow execution failed with exception:", e)

WorkflowEvent(type='started')
WorkflowEvent(type='status', state=IN_PROGRESS)
WorkflowEvent(type='superstep_started', iteration=1)
WorkflowEvent(type='executor_invoked', executor_id='start_node', data='Something short about a hot dog')
WorkflowEvent(type='output', executor_id='start_node', data='Starting execution')
WorkflowEvent(type='executor_completed', executor_id='start_node', data=['Something short about a hot dog', 'Starting execution'])
WorkflowEvent(type='superstep_completed', iteration=1)
WorkflowEvent(type='superstep_started', iteration=2)
WorkflowEvent(type='executor_invoked', executor_id='Poem Agent', data='Something short about a hot dog')
WorkflowEvent(type='executor_invoked', executor_id='Lyric Agent', data='Something short about a hot dog')
WorkflowEvent(type='output', executor_id='Lyric Agent', data=<agent_framework._types.AgentResponseUpdate object at 0x000002BBFA387890>)
WorkflowEvent(type='output', executor_id='Lyric Agent', data=<agent_framework._types.AgentRespon

In [26]:
log_capture.records

[<LogRecord: agent_framework._workflows._validation, 20, c:\Users\JOSHUAS\dev\josh-weston\microsoft-agent-framework-samples\.venv\Lib\site-packages\agent_framework\_workflows\_validation.py, 425, "Dead-end executors detected (no outgoing edges): ['reverse_executor']. Verify these are intended as final nodes in the workflow.">,
 <LogRecord: agent_framework._workflows._runner, 20, c:\Users\JOSHUAS\dev\josh-weston\microsoft-agent-framework-samples\.venv\Lib\site-packages\agent_framework\_workflows\_runner.py, 124, "Starting superstep 1">,
 <LogRecord: agent_framework._workflows._runner, 20, c:\Users\JOSHUAS\dev\josh-weston\microsoft-agent-framework-samples\.venv\Lib\site-packages\agent_framework\_workflows\_runner.py, 164, "Completed superstep 1">,
 <LogRecord: agent_framework._workflows._runner, 20, c:\Users\JOSHUAS\dev\josh-weston\microsoft-agent-framework-samples\.venv\Lib\site-packages\agent_framework\_workflows\_runner.py, 124, "Starting superstep 2">,
 <LogRecord: agent_framework._w

In [36]:
json_serialized = []
with open("events.jsonl", "w") as f:
    for event in events:
        f.write(json.dumps(make_serializable(event)) + "\n")

len(events), len(json_serialized)

found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__dict__
dict
to_json
found method: to_dict
__

(112, 0)

In [28]:
json_serialized

['{"type": "started", "data": null, "origin": "FRAMEWORK", "state": null, "details": null, "executor_id": null, "_request_id": null, "_source_executor_id": null, "_request_type": null, "_response_type": null, "iteration": null}',
 '{"type": "status", "data": null, "origin": "FRAMEWORK", "state": "IN_PROGRESS", "details": null, "executor_id": null, "_request_id": null, "_source_executor_id": null, "_request_type": null, "_response_type": null, "iteration": null}',
 '{"type": "superstep_started", "data": null, "origin": "EXECUTOR", "state": null, "details": null, "executor_id": null, "_request_id": null, "_source_executor_id": null, "_request_type": null, "_response_type": null, "iteration": 1}',
 '{"type": "executor_invoked", "data": "Something short about a hot dog", "origin": "FRAMEWORK", "state": null, "details": null, "executor_id": "start_node", "_request_id": null, "_source_executor_id": null, "_request_type": null, "_response_type": null, "iteration": null}',
 '{"type": "output",